Hugging Face transformers 라이브러리를 사용하여 문서 요약 모델을 구현하는 미션입니다. 데이터 로드 및 전처리부터 요약 모델 실행, 결과 평가까지 전체 파이프라인을 구축해 보세요.

In [2]:
import os

ROOT_DIR = os.getcwd()
is_colab_mode = False

# 코랩 모드
if ROOT_DIR == "/content":

    from google.colab import drive
    drive.mount('/content/drive')

    is_colab_mode = True


    print("[[ colab ]]")

    # import unicodedata

    DATA_DIR = os.path.join(ROOT_DIR, "data")
    RAW_DIR = os.path.join(DATA_DIR, "raw")

    if "raw.tar.gz" not in os.listdir():
        !wget https://github.com/wonbywondev/ML-DL/releases/download/data-v5/raw.tar.gz

    print("· 압축 파일 있음")


    if not os.path.exists(DATA_DIR):
        os.mkdir(DATA_DIR)

    if not os.path.exists(RAW_DIR):
        !tar -xzvf raw.tar.gz -C /content/data

    print("· data 압축 해제 완료")


    # train_json_path = unicodedata.normalize("NFC", train_json_path)
    # val_json_path = unicodedata.normalize("NFC", val_json_path)

# 로컬 모드
else:
    print("[[ local ]]")
    ROOT_DIR = "/".join(ROOT_DIR.split("/")[:-1])
    DATA_DIR = os.path.join(ROOT_DIR, "data")
    RAW_DIR = os.path.join(DATA_DIR, "raw")



train_edit_json_path = os.path.join(RAW_DIR, "train_original_editorial.json")
train_law_json_path = os.path.join(RAW_DIR, "train_original_news.json")
train_news_json_path = os.path.join(RAW_DIR, "train_original_law.json")
val_edit_json_path = os.path.join(RAW_DIR, "valid_original_editorial.json")
val_law_json_path = os.path.join(RAW_DIR, "valid_original_news.json")
val_news_json_path = os.path.join(RAW_DIR, "valid_original_law.json")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[[ colab ]]
· 압축 파일 있음
· data 압축 해제 완료


In [3]:
# 기타 환경 설정
import torch
import matplotlib.pyplot as plt
import gc


# 시각화 관련 설정
if not is_colab_mode:
    import matplotlib.font_manager as fm
    try:
        plt.rcParams['font.family'] = 'Apple SD Gothic Neo'
    except:
        try:
            plt.rcParams['font.family'] = 'NanumGothic'
        except:
            plt.rcParams['font.family'] = 'AppleGothic'

    plt.rcParams['axes.unicode_minus'] = False
    fm._load_fontmanager(try_read_cache=False)


# 디바이스 설정
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device("mps") # 맥 GPU
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda") # 윈도우 GPU
else:
    DEVICE = torch.device("cpu") # CPU


# 캐시 지우기 함수 생성
def clean_cache():
    gc.collect()
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()


# MallocStackLogging 에러 출력 방지
os.environ.pop("MallocStackLogging", None)
os.environ.pop("MallocStackLoggingNoCompact", None)
os.environ.pop("DYLD_INSERT_LIBRARIES", None)

In [ ]:
import json

with open(train_edit_json_path, "r", encoding="utf-8") as f:
    raw_train_edit = json.load(f)

In [ ]:
raw_train_edit

- highlight_indices 항목은 불용어의 인덱스를 표시해둔 리스트이다. 이는 해석 과정에서 유용하게 쓰이는 정보이므로 여기서는 필요가 없다.

In [ ]:
def json_to_dict(json_path: str):
    with open(json_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    RESULT_DICT = dict()

    docu_dict_list = raw_data["documents"]

    for result_dict_key, docu_dict in enumerate(docu_dict_list):

        text_dict_list = docu_dict["text"]
        target_index_list = docu_dict["extractive"]
        target = docu_dict["abstractive"][0]

        RESULT_DICT[result_dict_key] = {
            "text": list(),
            "full_text": "",
            "text_index": list(),
            "target_index": target_index_list,
            "target": target
            }

        for paragraph in text_dict_list:
            for sentence_dict in paragraph:
                sentence = sentence_dict["sentence"]
                RESULT_DICT[result_dict_key]["text"].append(sentence)
                RESULT_DICT[result_dict_key]["text_index"].append(sentence_dict["index"])

        RESULT_DICT[result_dict_key]["full_text"] = " ".join(RESULT_DICT[result_dict_key]["text"])

    return RESULT_DICT

In [ ]:
TRAIN_EDIT_DICT = json_to_dict(train_edit_json_path)

In [ ]:
TRAIN_EDIT_DICT[0]

### 결측치·중복치

In [ ]:
import pandas as pd

def drop_err(dictionary: dict):

    # 결측치: target_index
    tmp_df = (pd.DataFrame.from_dict(dictionary, orient='index').sort_index()).drop(["text", "text_index"], axis=1)
    target_idx_err_count = sum(tmp_df['target_index'].isna())

    for nan_key in tmp_df[tmp_df["target_index"].isna()].index:
        del dictionary[nan_key]

    tmp_df = tmp_df.drop(["target_index"], axis=1)


    err_target_set = set()
    for key, dict_ in dictionary.items():
        try:
            dict_["target_index"] = sorted(dict_["target_index"])
        except:
            target_idx_err_count += 1
            err_target_set.add(key)


    for err_key in err_target_set:
        del dictionary[err_key]

    print(f"· 결측치(주요 문장 인덱스): {target_idx_err_count}개")


    # 결측치: text_index
    nan_count = 0
    nan_set = set()

    for key, dict_ in dictionary.items():
        for i in range(len(dict_["text_index"])):
            if i not in dict_["text_index"]:
                nan_count += 1
                continue
        del dict_["text_index"]

    for nan_key in nan_set:
        del dictionary[nan_key]


    # 중복치
    for dup_key in tmp_df[tmp_df.duplicated()].index:
        del dictionary[dup_key]


    dictionary = {i: value for i, value in enumerate(dictionary.values())}

    print(f"· 결측치(누락 문장): {nan_count}개")
    print(f"· 중복치 {sum(tmp_df.duplicated())}개")
    print(f"· 수정 후 데이터: {len(dictionary)}개")


    return dictionary

In [ ]:
print("[editorial 학습 데이터]")
TRAIN_EDIT_DICT = drop_err(TRAIN_EDIT_DICT)
print("[editorial 검증 데이터]")
VAL_EDIT_DICT = json_to_dict(val_edit_json_path)
print(f"· 데이터: {len(VAL_EDIT_DICT)}개\n")


print("[law 학습 데이터]")
TRAIN_LAW_DICT = drop_err(json_to_dict(train_law_json_path))
print("[law 검증 데이터]")
VAL_LAW_DICT = json_to_dict(val_law_json_path)
print(f"· 데이터: {len(VAL_LAW_DICT)}개\n")


print("[news 학습 데이터]")
TRAIN_NEWS_DICT = drop_err(json_to_dict(train_news_json_path))
print("[news 검증 데이터]")
VAL_NEWS_DICT = json_to_dict(val_news_json_path)
print(f"· 데이터: {len(VAL_NEWS_DICT)}개")

In [ ]:
TRAIN_NEWS_DICT[0]

- law 데이터만 너무 많다. 균형 있게 섞어서 넣는 게 낫지 않을까?

In [ ]:
from datasets import Dataset, DatasetDict

edit_dataset = DatasetDict({
    "train": Dataset.from_list(list(TRAIN_EDIT_DICT.values())),
    "val": Dataset.from_list(list(VAL_EDIT_DICT.values()))
})

law_dataset = DatasetDict({
    "train": Dataset.from_list(list(TRAIN_LAW_DICT.values())),
    "val": Dataset.from_list(list(VAL_LAW_DICT.values()))
})

news_dataset = DatasetDict({
    "train": Dataset.from_list(list(TRAIN_NEWS_DICT.values())),
    "val": Dataset.from_list(list(VAL_NEWS_DICT.values()))
})

In [ ]:
news_dataset

# Hugging Face Dataset 변환 및 전처리

In [ ]:
import torch
from transformers import BartForConditionalGeneration
from transformers import PreTrainedTokenizerFast

model = BartForConditionalGeneration.from_pretrained('gogamza/kobart-summarization')
tokenizer = PreTrainedTokenizerFast.from_pretrained('gogamza/kobart-summarization')

In [ ]:
# def tokenize_function(example):
#     model_inputs = tokenizer(
#         example["full_text"],
#         max_length=512,
#         truncation=True
#     )
#     labels = tokenizer(
#         text_target=example["target"],
#         max_length=256,
#         truncation=True
#     )
#     model_inputs["labels"] = labels["input_ids"]
#     return model_inputs

In [ ]:
# from transformers import DataCollatorForSeq2Seq
# from transformers import TrainingArguments, Trainer

# tokenized_datasets = news_dataset.map(tokenize_function, batched=True)
# data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer)

# training_args = TrainingArguments(output_dir="test-trainer", report_to="none")

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_datasets["train"],
#     eval_dataset=tokenized_datasets["val"],
#     data_collator=data_collator,
#     tokenizer=tokenizer,
#     )

# trainer.train()

## 실험 모드 통합 및 체크포인트 관리
- 추출/생성 모드를 하나의 설정으로 제어하고, rough-1 평가와 시각화를 동일한 헬퍼로 처리합니다.
- 체크포인트는 Google Drive 경로만 바꿔주면 되도록 기본 변수를 선언했습니다.


In [ ]:
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

import evaluate
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, concatenate_datasets
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    DataCollatorWithPadding,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    Trainer,
    TrainingArguments,
    )

EXPERIMENT_DRIVE_ROOT = "/content/drive/Othercomputers/내 Mac/data/runs"  # TODO: 환경에 맞는 경로로 수정
# EXPERIMENT_DRIVE_ROOT = "/content/drive/MyDrive/summarization_runs"  # TODO: 환경에 맞는 경로로 수정
Path(EXPERIMENT_DRIVE_ROOT).mkdir(parents=True, exist_ok=True)

@dataclass
class ExperimentConfig:
    mode: str = "baseline"          # baseline | extractive | hybrid
    dataset: str = "news"           # edit | law | news | all
    experiment_name: str = "news_baseline"
    drive_root: str = EXPERIMENT_DRIVE_ROOT
    resume: bool = True
    include_extractive: bool = False
    num_train_epochs: int = 1
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 1
    learning_rate: float = 5e-5
    generation_max_length: int = 128
    save_total_limit: int = 2
    extractive_model_name: str = "klue/bert-base"
    generative_model_name: str = "gogamza/kobart-summarization"

AVAILABLE_DATASET_DICTS: Dict[str, DatasetDict] = {
    "edit": edit_dataset,
    "law": law_dataset,
    "news": news_dataset,
}
AVAILABLE_DATASET_DICTS["all"] = DatasetDict({
    split: concatenate_datasets([
        edit_dataset[split],
        law_dataset[split],
        news_dataset[split],
    ])
    for split in edit_dataset.keys()
})

rouge_metric = evaluate.load("rouge")



In [ ]:
def resolve_experiment_dir(config: ExperimentConfig, sub_dir: str = "") -> Path:
    base_dir = Path(config.drive_root) / config.experiment_name
    if sub_dir:
        base_dir = base_dir / sub_dir
    base_dir.mkdir(parents=True, exist_ok=True)
    return base_dir


def latest_checkpoint_path(output_dir: Path) -> Optional[str]:
    checkpoints = sorted(
        output_dir.glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1,
        )
    return str(checkpoints[-1]) if checkpoints else None


def log_metrics_to_csv(save_path: Path, metrics: Dict):
    save_path.parent.mkdir(parents=True, exist_ok=True)
    row = {**metrics, "timestamp": datetime.now().isoformat()}
    df = pd.DataFrame([row])
    if save_path.exists():
        existing = pd.read_csv(save_path)
        df = pd.concat([existing, df], ignore_index=True)
    df.to_csv(save_path, index=False)


def add_extractive_fields(example: Dict) -> Dict:
    target_index = example.get("target_index") or []
    sentences = example.get("text") or []
    chosen = [sentences[i] for i in target_index if i < len(sentences)]
    if not chosen and sentences:
        chosen = sentences[:1]
    return {
        "extract_summary": " ".join(chosen),
        "extract_sentence_count": len(chosen),
        }


def prepare_dataset(config: ExperimentConfig, add_extractive: bool = False) -> DatasetDict:
    dataset = AVAILABLE_DATASET_DICTS[config.dataset]
    if add_extractive:
        dataset = dataset.map(
            add_extractive_fields,
            desc=f"[{config.mode}] 추출 요약 필드 생성 ({config.dataset})",
            )
    return dataset



In [ ]:
def build_sentence_level_dataset(dataset: Dataset) -> Dataset:
    sentences, labels = [], []
    for sample in dataset:
        idx_set = set(sample.get("target_index") or [])
        for idx, sentence in enumerate(sample.get("text") or []):
            sentences.append(sentence)
            labels.append(1 if idx in idx_set else 0)
    return Dataset.from_dict({"sentence": sentences, "label": labels})


def tokenize_sentence_batch(batch, tokenizer):
    return tokenizer(batch["sentence"], truncation=True, padding=False, max_length=256)


def compute_extractive_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    labels = labels.astype(int)
    accuracy = (preds == labels).mean()
    tp = ((preds == 1) & (labels == 1)).sum()
    fp = ((preds == 1) & (labels == 0)).sum()
    fn = ((preds == 0) & (labels == 1)).sum()
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }


def train_extractive_model(config: ExperimentConfig) -> Dict:
    dataset = prepare_dataset(config, add_extractive=True)
    sentence_ds = DatasetDict({
        split: build_sentence_level_dataset(dataset[split])
        for split in dataset.keys()
    })

    tokenizer = AutoTokenizer.from_pretrained(config.extractive_model_name)
    tokenized = DatasetDict({
        split: sentence_ds[split].map(
            lambda batch: tokenize_sentence_batch(batch, tokenizer),
            batched=True,
            remove_columns=["sentence"],
            desc=f"[{config.mode}] 문장 토크나이징 ({split})",
        )
        for split in sentence_ds.keys()
    })

    data_collator = DataCollatorWithPadding(tokenizer)
    model = AutoModelForSequenceClassification.from_pretrained(
        config.extractive_model_name,
        num_labels=2,
    )

    output_dir = resolve_experiment_dir(config, "extractive")
    training_args = TrainingArguments(
        output_dir=str(output_dir),
        learning_rate=config.learning_rate,
        per_device_train_batch_size=config.per_device_train_batch_size,
        per_device_eval_batch_size=config.per_device_train_batch_size,
        num_train_epochs=config.num_train_epochs,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        evaluation_strategy="steps",
        save_strategy="steps",
        logging_steps=50,
        save_steps=200,
        eval_steps=200,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=config.save_total_limit,
        report_to=["none"],
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized.get("train"),
        eval_dataset=tokenized.get("val"),
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_extractive_metrics,
    )

    resume_ckpt = latest_checkpoint_path(output_dir) if config.resume else None
    trainer.train(resume_from_checkpoint=resume_ckpt)

    eval_metrics = trainer.evaluate()
    log_metrics_to_csv(output_dir / "metrics.csv", eval_metrics)
    return {
        "trainer": trainer,
        "tokenizer": tokenizer,
        "datasets": sentence_ds,
        "tokenized": tokenized,
        "metrics": eval_metrics,
    }



In [ ]:
def build_generative_tokenizer(config: ExperimentConfig):
    from transformers import BartForConditionalGeneration, PreTrainedTokenizerFast

    model = BartForConditionalGeneration.from_pretrained(config.generative_model_name)
    tok = PreTrainedTokenizerFast.from_pretrained(config.generative_model_name)
    return model, tok


def tokenize_with_mode(examples, tokenizer, mode: str):
    sources = []
    texts = examples["full_text"]
    summaries = examples.get("extract_summary")
    for idx, text in enumerate(texts):
        source_text = text
        if mode == "hybrid" and summaries is not None:
            extract = summaries[idx]
            if extract:
                source_text = f"[추출요약] {extract} </s> [본문] {text}"
        sources.append(source_text)

    model_inputs = tokenizer(
        sources,
        max_length=512,
        truncation=True,
    )

    labels = tokenizer(
        text_target=examples["target"],
        max_length=256,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


def tokenize_generative_dataset(dataset: DatasetDict, config: ExperimentConfig, tokenizer) -> DatasetDict:
    remove_columns = dataset["train"].column_names
    return DatasetDict({
        split: ds.map(
            lambda batch: tokenize_with_mode(batch, tokenizer, config.mode),
            batched=True,
            remove_columns=remove_columns,
            desc=f"[{config.mode}] 생성 토크나이징 ({config.dataset}-{split})",
        )
        for split, ds in dataset.items()
    })


def build_rouge_fn(tokenizer):
    def compute_metrics(eval_pred):
        preds, labels = eval_pred
        if isinstance(preds, tuple):
            preds = preds[0]
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        rouge = rouge_metric.compute(
            predictions=decoded_preds,
            references=decoded_labels,
            rouge_types=["rouge1", "rouge2", "rougeL"],
            use_stemmer=True,
        )
        return {
            "rough1": rouge["rouge1"],
            "rouge2": rouge["rouge2"],
            "rougeL": rouge["rougeL"],
        }

    return compute_metrics


def plot_training_history(trainer, save_path: Path):
    history_df = pd.DataFrame(trainer.state.log_history)
    if history_df.empty or "loss" not in history_df.columns:
        return
    fig, ax = plt.subplots(figsize=(6, 4))
    history_df = history_df.dropna(subset=["loss"])
    ax.plot(history_df["step"], history_df["loss"], label="loss")
    if "eval_loss" in history_df.columns:
        eval_df = history_df.dropna(subset=["eval_loss"])
        if not eval_df.empty:
            ax.plot(eval_df["step"], eval_df["eval_loss"], label="eval_loss")
    ax.set_xlabel("step")
    ax.set_ylabel("loss")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.close(fig)


def preview_generation_samples(trainer, tokenizer, raw_dataset: DatasetDict, split: str = "val", num_samples: int = 3):
    sample_count = min(num_samples, len(raw_dataset[split]))
    sample_dataset = raw_dataset[split].select(range(sample_count))
    generations = trainer.predict(sample_dataset).predictions
    if isinstance(generations, tuple):
        generations = generations[0]
    decoded_preds = tokenizer.batch_decode(generations, skip_special_tokens=True)
    preview = []
    for idx in range(sample_count):
        preview.append({
            "full_text": sample_dataset[idx]["full_text"][:200] + "...",
            "target": sample_dataset[idx]["target"],
            "prediction": decoded_preds[idx],
        })
    return pd.DataFrame(preview)



In [ ]:
def run_generative_experiment(config: ExperimentConfig) -> Dict:
    add_extractive = config.include_extractive or config.mode in {"extractive", "hybrid"}
    dataset = prepare_dataset(config, add_extractive=add_extractive)
    model, gen_tokenizer = build_generative_tokenizer(config)
    tokenized = tokenize_generative_dataset(dataset, config, gen_tokenizer)

    output_dir = resolve_experiment_dir(config, f"generative_{config.mode}")
    training_args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir),
        learning_rate=config.learning_rate,
        per_device_train_batch_size=config.per_device_train_batch_size,
        per_device_eval_batch_size=config.per_device_train_batch_size,
        num_train_epochs=config.num_train_epochs,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        predict_with_generate=True,
        generation_max_length=config.generation_max_length,
        evaluation_strategy="steps",
        save_strategy="steps",
        logging_steps=50,
        save_steps=200,
        eval_steps=200,
        save_total_limit=config.save_total_limit,
        report_to=["none"],
    )

    data_collator = DataCollatorForSeq2Seq(tokenizer=gen_tokenizer, model=model)
    compute_metrics = build_rouge_fn(gen_tokenizer)

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized.get("train"),
        eval_dataset=tokenized.get("val"),
        tokenizer=gen_tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    resume_ckpt = latest_checkpoint_path(output_dir) if config.resume else None
    trainer.train(resume_from_checkpoint=resume_ckpt)

    eval_metrics = trainer.evaluate(max_length=config.generation_max_length)
    log_metrics_to_csv(output_dir / "metrics.csv", eval_metrics)
    plot_training_history(trainer, output_dir / "history.png")

    return {
        "trainer": trainer,
        "tokenizer": gen_tokenizer,
        "dataset": dataset,
        "tokenized": tokenized,
        "metrics": eval_metrics,
        "preview": preview_generation_samples(trainer, gen_tokenizer, dataset),
        }



In [ ]:
# 실행 예시 (필요한 모드만 주석 해제하여 사용)
baseline_config = ExperimentConfig(
    mode="baseline",
    dataset="news",
    experiment_name="news_baseline",
    num_train_epochs=1,
)

hybrid_config = ExperimentConfig(
    mode="hybrid",
    dataset="news",
    experiment_name="news_hybrid",
    include_extractive=True,
    num_train_epochs=1,
)

# extractive_run = train_extractive_model(hybrid_config)
# baseline_run = run_generative_experiment(baseline_config)
# hybrid_run = run_generative_experiment(hybrid_config)